# FPP10 — Cross-Modal Matching with Frozen Encoders: only the mapping between the two spaces is learned (3 repetitions)

## The question
An existing system stores only **FingerJetFX minutiae templates**; new probes are **fingerprint images**. FPP10 asks the simplest form of the migration question: with **both trained encoders frozen as they are** — the image model (FPP04L winner) and the minutiae model (FPP09 winner) — can a **mapping** carry a stored template's embedding into the image space, so it can be compared with the probe image's embedding?

**Cross-modal test (decision set):** for a pair (a, b), **a = enrolled minutiae template** (minutiae model → mapping), **b = probe image** (image model); same pairs, protocol, sources (DB1, DB2, DB1×DB2) and threshold rule as every kernel. Both encoders are frozen: every arm works on their **saved embeddings**.

## Arms (mapping families)
| Arm | How the mapping is obtained | What it tests |
|---|---|---|
| **`frozen_as_is`** (baseline) | no mapping: cosine between the two raw embeddings | are the spaces related at all? (expected chance: the models never saw each other) |
| **`ridge`** | closed-form ridge regression minutiae embedding → image embedding **of the same capture** (training templates); α chosen on validation | the best linear map in least squares, no training loop |
| **`procrustes`** | closed-form best **rotation** (orthogonal Procrustes, SVD) between the two spaces | do the spaces differ only by a rotation? |
| **`linear_map`** | linear map (1024 → 1024) trained with the cross-modal loss | a linear map optimised for verification, not regression |
| **`mlp_adapter`** | small non-linear network (1024 → 2048 → 1024) trained with the cross-modal loss | is a non-linear map needed? |
| **`two_tower`** | small adapters on **both** sides meeting in a shared space (cross-modal loss) | a **migration matcher**: the image adapter is used only for the first login after migration (probe image vs migrated template); on a match the user is re-enrolled with the normal image embedding, so the image model and image templates never change |
| **`xgboost`** | gradient-boosted trees scoring each (minutiae embedding, image embedding) pair as same / different finger (features: element-wise product and absolute difference) | a strong non-neural baseline; gives a **score, not a migrated embedding** (it cannot be used with cosine after migration) |

`frozen_as_is`, `linear_map` and `mlp_adapter` were run first and are **reused from their saved outputs, not re-run**. (The earlier `finetune_minutiae` run belongs to FPP11 — encoder fine-tuning — and is reused there.)

## How the migration is deployed (why both-side adapters are allowed)
1. **Batch migration, once:** every stored minutiae template → (minutiae-side mapping) → stored migration template.
2. **First login:** probe image → frozen image model → (image-side adapter, `two_tower` only) → cosine with the migration template.
3. **On a match:** the user is re-enrolled with the **normal image embedding**; the migration template and the adapters are no longer used.
4. **Afterwards:** image vs image with the unchanged image model. So only the first-login cross-modal match is decided here.

## Training of the trained arms
- Cross-modal **BCE** on (minutiae a, image b) and (minutiae b, image a), genuine and impostor pairs (learned calibration) **+ a pull** of each template's embedding towards the image embedding **of the same capture** (λ = 1.0); source-balanced sampler, 64 pairs per batch, 12,000 pairs per epoch; AdamW 1e-3, weight decay 1e-4, clip 1.0, ≤ 100 epochs; early stopping / LR drops / best epoch on the **cross-modal source-balanced validation EER** (patience 10, LR × 0.3 after 3).
- `ridge` α and the number of `xgboost` trees are chosen by the same validation EER.

## What is reported
- **Decision — validation only:** winner = lowest mean cross-modal source-balanced validation EER (ties within 0.01 → higher AUC).
- **Reference rows:** image vs image (FPP04L) = the **ceiling**; minutiae vs minutiae (FPP09) and FingerJetFX + BOZORTH3 = the **old system**.
- **Test and FingerDOS — reported for every row incl. F1, never used to decide.**
- Reported only: minutiae vs minutiae after mapping, the reverse direction (image enrolled, minutiae probe), the **migration matrix** of the winner, the cosine between a template and the image of the same capture; bias check with the 5 calibration subjects.

**Hardware:** any GPU (A100 fine); every arm runs in minutes. **Needs:** FPP05 manifest (image winner), FPP09 finished (minutiae winner), fpp_lib ≥ 1.10.

In [ ]:
%pip install -q openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load the shared libraries from output/00_code/ (fpf_lib from FPF00, fpp_lib installed by FPP01), so every FPP kernel uses the same version.
import shutil, sys, importlib
from pathlib import Path
def _find_folder(name, root=Path('/content/drive/MyDrive'), max_depth=4):
    frontier = [root]
    for _ in range(max_depth + 1):
        nxt = []
        for d in frontier:
            try: kids = [c for c in d.iterdir() if c.is_dir()]
            except OSError: continue
            for c in kids:
                if c.name == name: return c
                nxt.append(c)
        frontier = nxt
    return None
ROOT = _find_folder('FingerPrint-Final')
if ROOT is None: raise FileNotFoundError("'FingerPrint-Final' not found under My Drive")
CODE = ROOT / 'output' / '00_code'
for m in ('fpf_lib.py', 'fpf_tf.py', 'fpp_lib.py'):
    src = CODE / m
    if not src.is_file(): raise FileNotFoundError(f'{src} missing: run ' + ('FPP01' if m == 'fpp_lib.py' else 'FPF00') + ' first')
    shutil.copy2(src, Path('/content') / m)
sys.path.insert(0, '/content')
import fpf_lib as L; importlib.reload(L)
import fpf_tf as T; importlib.reload(T)
import fpp_lib as P; importlib.reload(P)
import numpy as np, pandas as pd, json, time, os, gc
if tuple(int(x) for x in P.VERSION.split('.')) < (1, 10): raise RuntimeError(f'fpp_lib {P.VERSION} in output/00_code is too old: upload fpp_lib 1.10')
ENV = L.env_info(); print('fpp_lib', P.VERSION, '|', json.dumps(ENV, indent=1))

## 1. Configuration, the frozen image model (FPP04L winner) and the frozen minutiae model (FPP09 winner)

In [ ]:
import h5py, math, subprocess, tensorflow as tf
import matplotlib.pyplot as plt
KERNEL = 'FPP10'; ws = L.workspace(); KDIR = P.kernel_dir(ws, KERNEL); FIG = KDIR / 'figures'
REPS = L.repetitions(ws); REPS_TO_RUN = REPS
ARMS = ['frozen_as_is', 'ridge', 'procrustes', 'linear_map', 'mlp_adapter', 'two_tower', 'xgboost']
ARMS_TO_RUN = ARMS                     # finished runs are skipped (frozen_as_is, linear_map, mlp_adapter were run first)
LAMBDA_PULL = 1.0; ADAPTER_HIDDEN = 2048; ADAPTER_DROPOUT = 0.1
LR_BY_ARM = {'linear_map': 1e-3, 'mlp_adapter': 1e-3, 'two_tower': 1e-3}; EPOCHS_BY_ARM = {'linear_map': 100, 'mlp_adapter': 100, 'two_tower': 100}
RIDGE_ALPHAS = [1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0, 1000.0]
XGB = dict(n_estimators=1500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.5, tree_method='hist', device='cuda', eval_metric='logloss')
XGB_TRAIN_PAIRS = 200_000; XGB_TREE_GRID = [100, 200, 300, 500, 750, 1000, 1250, 1500]
WEIGHT_DECAY = 1e-4; CLIPNORM = 1.0; PATIENCE = 10; LR_PATIENCE = 3; LR_FACTOR = 0.3
PAIR_BATCH = 64; TRAIN_PAIRS_PER_EPOCH = 12000; VAL_PAIRS = None; EMBED_BATCH = 512; TIMING_REPEATS = 30; TRAIN_AUGMENTATION = True
QUICK_SMOKE_TEST = False
if QUICK_SMOKE_TEST:
    EPOCHS_BY_ARM = {a: 2 for a in EPOCHS_BY_ARM}; XGB['n_estimators'] = 50; XGB_TREE_GRID = [25, 50]; XGB_TRAIN_PAIRS = 5000; TRAIN_PAIRS_PER_EPOCH = 384; VAL_PAIRS = 384; REPS_TO_RUN = REPS[:1]; KDIR = KDIR / '_smoke_test'; FIG = KDIR / 'figures'
FIG.mkdir(parents=True, exist_ok=True); (KDIR / 'runs').mkdir(parents=True, exist_ok=True); LOG = L.RunLog(KDIR / 'FPP10_run_log.csv')
tf.keras.mixed_precision.set_global_policy('float32')
MOD_PATH = Path('/content/fpp10_minutiae.py'); MOD_PATH.write_text(r'''# fpp10_minutiae.py — the FPP09 minutiae module, unchanged (FPP10 maps its embeddings into the image space)
import numpy as np
SETUP = dict(quality=False, k=16, depth=2, N=128, aggregate='max', crop=False)          # architecture ablation: pilot defaults (FPM07), no quality input
ST = dict(d=256, heads=4, blocks=3, ff=512)                 # Set Transformer: model width, heads, self-attention blocks, feed-forward width
EDGE1, EDGE2, POINT = [64, 128], [256, 256], 512; EMBEDDING_DIM = 1024; DROPOUT = 0.3; GEO_SCALE = 4.0
AUG = dict(rotation_deg=5.0, shift_px=10.0, dropout=0.10, position_sigma_px=2.0, angle_sigma_deg=5.0, p_original=0.25, crop_area_min=0.70)
# placement (as the FPP01 image recipe: rotation +-5 deg, shift +-3 % of the image ~ +-10 px at 500 ppi) + FingerJetFX noise (FPM08: 10 % minutia dropout, 2 px / 5 deg jitter); 25 % of draws stay original (as FPP01 mix_versions)
COORD_HALF = 112.0      # native 500 ppi pixels, centred on the image, / 112 (one physical scale for every sensor)

def load_minutiae(h5file):
    import h5py
    out = []
    with h5py.File(h5file, 'r') as h:
        g = h['minutiae']; X, Y, TH, TY, Q = g['x'][:], g['y'][:], g['theta'][:], g['type'][:], g['quality'][:]; W, H = g['image_width'][:], g['image_height'][:]
        PPI = g['ppi'][:] if 'ppi' in g else np.full(len(X), 500)
    for i in range(len(X)):
        f = 500.0 / float(PPI[i] or 500)
        out.append({'x': ((np.asarray(X[i], np.float32) - W[i] / 2) * f).astype(np.float32), 'y': ((np.asarray(Y[i], np.float32) - H[i] / 2) * f).astype(np.float32),
                    'theta': np.deg2rad(np.asarray(TH[i], np.float32)).astype(np.float32), 'type': np.clip(np.asarray(TY[i], np.int64), 0, 2),
                    'quality': np.asarray(Q[i], np.float32)})
    return out

def augment(m, rng, crop=False):
    # training only: with p 0.25 the original; else whole-print rotation + shift (placement), then minutia dropout and position / angle jitter (extraction noise)
    if len(m['x']) == 0 or rng.random() < AUG['p_original']: return m
    m = rotate_translate(m, np.deg2rad(rng.uniform(-AUG['rotation_deg'], AUG['rotation_deg'])), rng.uniform(-AUG['shift_px'], AUG['shift_px']), rng.uniform(-AUG['shift_px'], AUG['shift_px']))
    x, y, t, ty, q = m['x'].copy(), m['y'].copy(), m['theta'].copy(), m['type'].copy(), m['quality'].copy()
    if crop and len(x) >= 8:
        # partial crop (FPM08): a random window covering 70-100 % of the area the minutiae occupy; minutiae outside it are removed (kept only if >= 5 remain)
        x0, x1, y0, y1 = x.min(), x.max(), y.min(), y.max(); s = np.sqrt(rng.uniform(AUG['crop_area_min'], 1.0)); w, h = (x1 - x0) * s, (y1 - y0) * s
        cx, cy = rng.uniform(x0, x1 - w) if x1 - x0 > w else x0, rng.uniform(y0, y1 - h) if y1 - y0 > h else y0
        inside = (x >= cx) & (x <= cx + w) & (y >= cy) & (y <= cy + h)
        if inside.sum() >= 5: x, y, t, ty, q = x[inside], y[inside], t[inside], ty[inside], q[inside]
    keep = rng.random(len(x)) >= AUG['dropout']
    if keep.sum() >= 5: x, y, t, ty, q = x[keep], y[keep], t[keep], ty[keep], q[keep]
    k = len(x); x = x + rng.normal(0, AUG['position_sigma_px'], k).astype(np.float32); y = y + rng.normal(0, AUG['position_sigma_px'], k).astype(np.float32)
    t = t + rng.normal(0, np.deg2rad(AUG['angle_sigma_deg']), k).astype(np.float32)
    return {'x': x.astype(np.float32), 'y': y.astype(np.float32), 'theta': t.astype(np.float32), 'type': ty, 'quality': q}

def to_set(m, n_max=128, quality=False):
    # (n_max, F + 1): x, y (/ COORD_HALF), cos, sin, type one-hot (3), [quality], mask; extra minutiae dropped by lowest quality
    F_ = 8 if quality else 7; n = len(m['x']); keep = np.argsort(-m['quality'], kind='stable')[:n_max] if n > n_max else np.arange(n); k = len(keep)
    s = np.zeros((n_max, F_ + 1), np.float32)
    s[:k, 0] = m['x'][keep] / COORD_HALF; s[:k, 1] = m['y'][keep] / COORD_HALF; s[:k, 2] = np.cos(m['theta'][keep]); s[:k, 3] = np.sin(m['theta'][keep])
    s[np.arange(k), 4 + m['type'][keep]] = 1.0
    if quality: s[:k, 7] = m['quality'][keep]
    s[:k, F_] = 1.0; return s

def rotate_translate(m, angle, tx, ty):
    c, s = np.cos(angle), np.sin(angle); x, y = m['x'], m['y']
    return {**m, 'x': (c * x - s * y + tx).astype(np.float32), 'y': (s * x + c * y + ty).astype(np.float32), 'theta': (m['theta'] + angle).astype(np.float32)}

def build_ls_encoder(cfg=SETUP, emb_dim=EMBEDDING_DIM):
    # local structure (FPM07 / FPM08): each minutia described by its k nearest neighbours in its own frame -> invariant to rotation and translation by design
    import tensorflow as tf
    Lyr = tf.keras.layers
    class MaskedPool(Lyr.Layer):
        def call(self, inputs):
            h, m = inputs; m3 = tf.cast(m, h.dtype)[..., None]; cnt = tf.reduce_sum(m3, 1)
            mean = tf.reduce_sum(h * m3, 1) / tf.maximum(cnt, 1.0); mx = tf.reduce_max(h + (m3 - 1.0) * 1e9, 1)
            return tf.concat([tf.where(cnt > 0, mx, tf.zeros_like(mx)), mean], -1)
    class KNN(Lyr.Layer):
        def __init__(self, k, **kw): super().__init__(**kw); self.k = int(k)
        def call(self, inputs):
            x, m = inputs; px, py = x[..., 0], x[..., 1]; n = tf.shape(x)[1]
            d2 = (px[:, None, :] - px[:, :, None]) ** 2 + (py[:, None, :] - py[:, :, None]) ** 2
            d2 = d2 + (1.0 - m[:, None, :]) * 1e9 + tf.eye(n, dtype=d2.dtype)[None] * 2e9
            return tf.math.top_k(-d2, k=self.k).indices
    class EdgeGeometry(Lyr.Layer):
        def __init__(self, quality=False, **kw): super().__init__(**kw); self.quality = bool(quality)
        def call(self, inputs):
            x, m, idx = inputs; xj = tf.gather(x, idx, batch_dims=1); xi = x[:, :, None, :]
            dx = xj[..., 0] - xi[..., 0]; dy = xj[..., 1] - xi[..., 1]; ci, si, cj, sj = xi[..., 2], xi[..., 3], xj[..., 2], xj[..., 3]
            u = dx * ci + dy * si; v = -dx * si + dy * ci; dist = tf.sqrt(u * u + v * v + 1e-12)
            ti = tf.broadcast_to(xi[..., 4:7], tf.shape(xj[..., 4:7])); valid = tf.gather(m, idx, batch_dims=1) * m[:, :, None]
            parts = [tf.stack([GEO_SCALE * u, GEO_SCALE * v, GEO_SCALE * dist, cj * ci + sj * si, sj * ci - cj * si], -1), ti, xj[..., 4:7]]
            if self.quality: parts += [tf.broadcast_to(xi[..., 7:8], tf.shape(xj[..., 7:8])), xj[..., 7:8]]
            return tf.concat(parts + [valid[..., None]], -1)
    class MaskedAttnK(Lyr.Layer):
        # neighbour attention: softmax over the k neighbours of a learned score per edge (padding masked); weighted sum of the edge features
        def call(self, inputs):
            e, s, valid = inputs; logits = s[..., 0] + (valid - 1.0) * 1e9; w = tf.nn.softmax(logits, axis=2)
            out = tf.reduce_sum(w[..., None] * e, 2); return tf.where(tf.reduce_sum(valid, 2, keepdims=True) > 0, out, tf.zeros_like(out))
    def aggregate(e, valid, name):
        if cfg.get('aggregate', 'max') == 'max': return MaskedMaxK(name=name)([e, valid])
        return MaskedAttnK(name=name)([e, Lyr.Dense(1, name=f'{name}_attn_score')(e), valid])
    class GatherNeighbours(Lyr.Layer):
        def call(self, inputs): h, idx = inputs; return tf.gather(h, idx, batch_dims=1)
    class MaskedMaxK(Lyr.Layer):
        def call(self, inputs):
            e, valid = inputs; v = valid[..., None]; mx = tf.reduce_max(e + (v - 1.0) * 1e9, 2)
            return tf.where(tf.reduce_sum(v, 2) > 0, mx, tf.zeros_like(mx))
    F_ = 8 if cfg['quality'] else 7; k = cfg['k']; inp = tf.keras.Input((cfg['N'], F_ + 1), dtype=tf.float32, name='minutiae_set')
    x = Lyr.Lambda(lambda z: z[..., :F_], name='features')(inp); m = Lyr.Lambda(lambda z: z[..., F_], name='mask')(inp)
    idx = KNN(k, name='knn')([x, m]); eg = EdgeGeometry(cfg['quality'], name='edge_geometry')([x, m, idx])
    geo = Lyr.Lambda(lambda z: z[..., :-1], name='edge_features')(eg); valid = Lyr.Lambda(lambda z: z[..., -1], name='edge_valid')(eg)
    e = geo
    for i, u in enumerate(EDGE1): e = Lyr.Dense(u, activation='relu', name=f'edge1_dense_{i}')(e)   # explicit names: weights load by structure in any process
    h = aggregate(e, valid, 'local_structure_1'); hs = [h]
    for layer in range(2, cfg['depth'] + 1):
        hj = GatherNeighbours(name=f'gather_{layer}')([h, idx]); hi = Lyr.Lambda(lambda z: tf.repeat(z[:, :, None, :], k, axis=2), name=f'tile_{layer}')(h)
        e2 = Lyr.Concatenate(name=f'edge_concat_{layer}')([hi, Lyr.Subtract(name=f'edge_diff_{layer}')([hj, hi]), geo])
        for i, u in enumerate(EDGE2): e2 = Lyr.Dense(u, activation='relu', name=f'edge{layer}_dense_{i}')(e2)
        h = aggregate(e2, valid, f'local_structure_{layer}'); hs.append(h)
    h = Lyr.Dense(POINT, activation='relu', name='point_dense')(Lyr.Concatenate(name='local_concat')(hs) if len(hs) > 1 else hs[0]); pooled = MaskedPool(name='pool')([h, m])
    f = Lyr.Dropout(DROPOUT, name='head_dropout')(pooled); e = Lyr.Dense(emb_dim, use_bias=False, name='embedding_dense')(f); e = Lyr.BatchNormalization(name='embedding_bn')(e)
    e = Lyr.Lambda(lambda z: tf.math.l2_normalize(z, axis=1), name='embedding')(e)
    return tf.keras.Model(inp, e, name='minutiae_local_structure_encoder')

def build_st_encoder(cfg=SETUP, st=ST, emb_dim=EMBEDDING_DIM):
    # Set Transformer (Lee et al. 2019): per-minutia features -> pre-norm self-attention blocks over all minutiae (padding masked) -> attention pooling (PMA, 1 seed) + masked mean
    import tensorflow as tf
    Lyr = tf.keras.layers
    class Seed(Lyr.Layer):
        def build(self, shape): self.s = self.add_weight(name='seed', shape=(1, 1, st['d']), initializer='glorot_uniform')
        def call(self, h): return tf.tile(self.s, [tf.shape(h)[0], 1, 1])
    class MaskedMean(Lyr.Layer):
        def call(self, inputs):
            h, m = inputs; m3 = m[..., None]; return tf.reduce_sum(h * m3, 1) / tf.maximum(tf.reduce_sum(m3, 1), 1.0)
    F_ = 8 if cfg['quality'] else 7; inp = tf.keras.Input((cfg['N'], F_ + 1), dtype=tf.float32, name='minutiae_set')
    x = Lyr.Lambda(lambda z: z[..., :F_], name='features')(inp); m = Lyr.Lambda(lambda z: z[..., F_], name='mask')(inp)
    key_mask = Lyr.Lambda(lambda z: tf.tile(z[:, None, :] > 0.5, [1, tf.shape(z)[1], 1]), name='key_mask')(m)       # (B, N, N): attend to real minutiae only
    seed_mask = Lyr.Lambda(lambda z: z[:, None, :] > 0.5, name='seed_mask')(m)                                       # (B, 1, N)
    h = Lyr.Dense(st['d'], name='input_proj')(x)
    for b in range(st['blocks']):
        a = Lyr.LayerNormalization(name=f'sab{b}_ln1')(h)
        a = Lyr.MultiHeadAttention(st['heads'], st['d'] // st['heads'], name=f'sab{b}_mha')(a, a, attention_mask=key_mask)
        h = Lyr.Add(name=f'sab{b}_res1')([h, a])
        f = Lyr.LayerNormalization(name=f'sab{b}_ln2')(h); f = Lyr.Dense(st['ff'], activation='gelu', name=f'sab{b}_ff1')(f); f = Lyr.Dense(st['d'], name=f'sab{b}_ff2')(f)
        h = Lyr.Add(name=f'sab{b}_res2')([h, f])
    h = Lyr.LayerNormalization(name='final_ln')(h)
    q = Seed(name='pma_seed')(h); pma = Lyr.MultiHeadAttention(st['heads'], st['d'] // st['heads'], name='pma_mha')(q, h, attention_mask=seed_mask)
    pooled = Lyr.Concatenate(name='pool')([Lyr.Flatten(name='pma_flat')(pma), MaskedMean(name='masked_mean')([h, m])])
    f = Lyr.Dropout(DROPOUT, name='head_dropout')(pooled); e = Lyr.Dense(emb_dim, use_bias=False, name='embedding_dense')(f); e = Lyr.BatchNormalization(name='embedding_bn')(e)
    e = Lyr.Lambda(lambda z: tf.math.l2_normalize(z, axis=1), name='embedding')(e)
    return tf.keras.Model(inp, e, name='minutiae_set_transformer_encoder')

BUILDERS = {'set_transformer': build_st_encoder, 'local_structure': build_ls_encoder}
def build_encoder(arch): return BUILDERS[arch]()

def arm_cfg(arm):
    # 'k16' (FPP06) / 'k8' / 'k24' / 'k32' / 'k<K>_attention'
    k = int(arm.split('_')[0][1:]); return dict(SETUP, k=k, aggregate='attention' if arm.endswith('_attention') else 'max')
def build_arm(arm): return build_ls_encoder(arm_cfg(arm))

def transfer(src, dst):
    # copy every weight of `src` into the layer of the same name in `dst`; a layer that only GREW its input (quality inputs, a third layer feeding the
    # concatenation) gets the old weights in its first rows and ZEROS in the new rows -> dst starts with exactly the same function as src
    names = {l.name: l for l in dst.layers}; copied, grown = [], []
    for l in src.layers:
        if not l.weights: continue
        d = names.get(l.name)
        if d is None: raise RuntimeError(f'layer {l.name} missing in the new model')
        ws_, wd = l.get_weights(), d.get_weights(); new = []
        for a, b in zip(ws_, wd):
            if a.shape == b.shape: new.append(a)
            elif a.ndim == 2 and b.ndim == 2 and a.shape[1] == b.shape[1] and b.shape[0] > a.shape[0]:
                z = np.zeros_like(b); z[:a.shape[0]] = a; new.append(z)
            else: raise RuntimeError(f'cannot transfer {l.name}: {a.shape} -> {b.shape}')
        d.set_weights(new); (grown if any(a.shape != b.shape for a, b in zip(ws_, wd)) else copied).append(l.name)
    fresh = [l.name for l in dst.layers if l.weights and l.name not in {s.name for s in src.layers}]
    return {'layers_copied': len(copied), 'layers_grown_zero_rows': grown, 'layers_new': fresh}
'''); sys.path.insert(0, '/content')
import fpp10_minutiae as M; importlib.reload(M)

def winner_dirs(kernel):
    # {rep: run dir} of a kernel's winner (from its manifest's file list) + the manifest
    mf = json.loads((P.kernel_dir(ws, kernel) / 'winner_manifest.json').read_text())
    return {r: (Path(ws['output']) / f['pair_scores.npz']).parent for r, f in mf['files'].items()}, mf
IMG_RUN, mf5 = winner_dirs('FPP05'); MIN_RUN, mf9 = winner_dirs('FPP09')
M_CFG = dict(M.SETUP, **mf9['config'])
IMG_LABEL = 'Image vs image (FPP04L winner, ceiling)'; MIN_LABEL = 'Minutiae vs minutiae (FPP09 winner, old template)'; BOZ = 'FingerJetFX + BOZORTH3 (FPP03)'
print('image model  :', {r[:6]: f'{d.parents[2].name}/{d.name}' for r, d in IMG_RUN.items()})
print('minutiae model:', {r[:6]: f'{d.parents[2].name}/{d.name}' for r, d in MIN_RUN.items()}, '| config', M_CFG)
CONFIG = dict(kernel=KERNEL, kernel_name=P.FPP_KERNELS[KERNEL], protocol=P.PROTOCOL_TABLE, arms=ARMS, image_model='FPP04L winner, frozen (saved embeddings of the clean images)',
              minutiae_model=f"FPP09 winner ({mf9['winning_arm']}), config {M_CFG}", cross_modal='pair (a, b): a = enrolled minutiae template, b = probe image',
              loss=f'cross-modal BCE both directions (learned calibration) + {LAMBDA_PULL} x (1 - cos(minutiae embedding, image embedding of the same capture))',
              adapter=dict(linear='Dense 1024 (no bias)', mlp=f'Dense {ADAPTER_HIDDEN} gelu, dropout {ADAPTER_DROPOUT}, Dense 1024', two_tower='mlp adapter on each side, shared 1024 space'),
              ridge=dict(alphas=RIDGE_ALPHAS, target='image embedding of the same capture', selection='cross-modal balanced validation EER'), procrustes='orthogonal map from SVD of M^T I (training templates)',
              xgboost=dict(XGB, features='m*i and |m-i| (2048)', train_pairs=XGB_TRAIN_PAIRS, trees_selected_by='cross-modal balanced validation EER'),
              lr=LR_BY_ARM, epochs=EPOCHS_BY_ARM, weight_decay=WEIGHT_DECAY, clipnorm=CLIPNORM, patience=PATIENCE, lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR,
              pair_batch=PAIR_BATCH, train_pairs_per_epoch=TRAIN_PAIRS_PER_EPOCH, sampler='source-balanced DB1 / DB2 / DB1xDB2',
              augmentation='none: both encoders are frozen; every arm works on their saved embeddings of the clean templates / images',
              early_stopping='cross-modal ' + P.SELECTION_RULE, threshold=P.THRESHOLD_RULE + ' (cross-modal scores)', decision=P.DECISION_RULE,
              local_runs=str(P.LOCAL_RUNS), repetitions=REPS, quick_smoke_test=QUICK_SMOKE_TEST, fpp_lib_version=P.VERSION)
print(json.dumps(CONFIG, indent=2, default=str))

## 2. Saved embeddings (image model, minutiae model), minutiae templates, protocol, cross-modal pair sequence
All embeddings share one index space: image *i* and minutiae template *i* come from the same capture.

In [ ]:
I_EMB = {r: {d: L.l2n(v.astype(np.float32)) for d, v in L.load_npz(IMG_RUN[r] / 'embeddings_float32.npz').items()} for r in REPS}       # frozen image embeddings
M0_EMB = {r: {d: L.l2n(v.astype(np.float32)) for d, v in L.load_npz(MIN_RUN[r] / 'embeddings_float32.npz').items()} for r in REPS}      # FPP09 minutiae embeddings
MINU = {d: M.load_minutiae(L.local_copy(ws['templates'] / f'{d}_fingerjetfx_templates.h5')) for d in L.DATASETS}
MOLF = ('MOLF_DB1', 'MOLF_DB2'); MINU_MOLF = MINU['MOLF_DB1'] + MINU['MOLF_DB2']; OFFSET = {'MOLF_DB1': 0, 'MOLF_DB2': len(MINU['MOLF_DB1'])}
SETS = {d: np.stack([M.to_set(m, quality=M_CFG['quality']) for m in MINU[d]]) for d in L.DATASETS}; SETS['MOLF'] = np.concatenate([SETS[d] for d in MOLF])
for r in REPS:
    for d in L.DATASETS:
        if not (len(I_EMB[r][d]) == len(M0_EMB[r][d]) == len(MINU[d])): raise RuntimeError(f'{r} {d}: image embeddings, minutiae embeddings and templates differ in count')
I_MOLF = {r: np.concatenate([I_EMB[r][d] for d in MOLF]) for r in REPS}; M0_MOLF = {r: np.concatenate([M0_EMB[r][d] for d in MOLF]) for r in REPS}
PROTO = {r: P.load_pairs(ws, r) for r in REPS}; print('FPP01 pair files verified for', len(REPS), 'repetitions')
IMG_MOLF = None                     # set per run: the minutiae-side inputs of the validation images (embeddings for adapters, sets for fine-tuning)
# how far apart are the spaces? cosine between the minutiae and the image embedding of the SAME capture, before any training
r0 = REPS[0]; same = np.sum(M0_MOLF[r0] * I_MOLF[r0], 1); rnd = np.sum(M0_MOLF[r0] * I_MOLF[r0][np.random.default_rng(0).permutation(len(I_MOLF[r0]))], 1)
print(f'frozen spaces ({r0[:6]}): cos(minutiae, image) same capture {same.mean():.3f} ± {same.std():.3f} | random pairs {rnd.mean():.3f} ± {rnd.std():.3f}')

class CrossPairSeq(tf.keras.utils.Sequence):
    # source-balanced pairs: minutiae-side input of a and b (adapter: frozen FPP09 embeddings; fine-tuning: freshly augmented sets) + the image embeddings of a and b
    def __init__(self, sampler, seed, kind, X_molf, I_molf, workers=4):
        super().__init__(workers=workers, use_multiprocessing=False, max_queue_size=8); self.s = sampler; self.seed = seed; self.kind = kind; self.X = X_molf; self.I = I_molf; self.epoch = 0
    def __len__(self): return len(self.s)
    def _x(self, idx, rng):
        if self.kind == 'encoder' and TRAIN_AUGMENTATION: return np.stack([M.to_set(M.augment(MINU_MOLF[int(i)], rng, M_CFG['crop']), quality=M_CFG['quality']) for i in idx])
        return self.X[np.asarray(idx)]
    def __getitem__(self, b):
        ia, ib, y = self.s.batch(b); ia, ib = np.asarray(ia), np.asarray(ib); rng = np.random.default_rng([self.seed, self.epoch, b])
        return (self._x(ia, rng), self._x(ib, rng), self.I[ia], self.I[ib]), y
    def on_epoch_end(self): self.s.on_epoch_end(); self.epoch += 1

## 3. Models: adapters, cross-modal training model, cross-modal validation callback
`CrossValidation` scores each validation pair as cos(minutiae model(template a), image embedding b) — the deployment direction.

In [ ]:
class EpochClock(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None): self.t0 = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None: logs['epoch_seconds'] = time.perf_counter() - self.t0
class BalancedEERCallback(tf.keras.callbacks.Callback):
    # after each epoch: embed the validation images, score all validation pairs, source-balanced EER -> best checkpoint / LR drop / early stop (fpp_lib rule)
    def __init__(self, encoder, bv, ckpt, hist_csv):
        super().__init__(); self.encoder = encoder; self.bv = bv; self.ckpt = ckpt; self.hist_csv = hist_csv; self.x = IMG_MOLF[bv.indices]; self.rows = []
        self.stop = P.BalancedEERStopping(PATIENCE, LR_PATIENCE, LR_FACTOR)
    def _lr(self): return float(np.asarray(self.model.optimizer.learning_rate))
    def _set_lr(self, v):
        try: self.model.optimizer.learning_rate.assign(v)
        except AttributeError: self.model.optimizer.learning_rate = v
    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}; t0 = time.perf_counter()
        m = self.bv.score(L.l2n(T.embed(self.encoder, self.x, EMBED_BATCH))); logs.update(m); lr = self._lr(); d = self.stop.update(epoch, m['val_balanced_eer'], m['val_balanced_auc'])
        if d['improved']: self.model.save_weights(str(self.ckpt))
        if d['reduce_lr'] and lr > self.stop.min_lr: self._set_lr(max(lr * LR_FACTOR, self.stop.min_lr))
        if d['stop']: self.model.stop_training = True
        logs['validation_seconds'] = time.perf_counter() - t0
        self.rows.append({'epoch': epoch + 1, **{k: float(np.asarray(v)) for k, v in logs.items() if np.asarray(v).size == 1}, 'lr': lr, 'improved': d['improved']}); pd.DataFrame(self.rows).to_csv(self.hist_csv, index=False)
        print(f"  epoch {epoch + 1}: val balanced EER {100 * m['val_balanced_eer']:.2f} % | AUC {100 * m['val_balanced_auc']:.2f} % | "
              + ' '.join(f"{n.split('_')[0]} {100 * m[f'val_eer_{n}']:.2f}" for n in P.VAL_SOURCES) + (' *best' if d['improved'] else '') + (' | LR down' if d['reduce_lr'] else '') + (' | stop' if d['stop'] else ''))


class CrossValidation(P.BalancedValidation):
    # validation pair (a, b): a = enrolled minutiae (its embedding from the mapping being trained), b = probe image (image embedding; two_tower: through the image adapter g)
    def __init__(self, Pp, offset, I_molf, max_pairs=None, g=None):
        super().__init__(Pp, offset, max_pairs=max_pairs); self.I_raw = np.asarray(I_molf, np.float32)[self.indices]; self.g = g; self.I = L.l2n(self.I_raw)
    def score(self, emb):
        if self.g is not None: self.I = L.l2n(T.embed(self.g, self.I_raw, EMBED_BATCH))          # the image tower changes during training
        return super().score(emb)
    def pair_score(self, E, ia, ib): return np.sum(E[ia] * self.I[ib], axis=1)

def build_adapter(arm, side='minutiae'):
    x = tf.keras.Input((M.EMBEDDING_DIM,), dtype=tf.float32, name=f'{side}_embedding')
    pre = '' if side == 'minutiae' else 'image_'          # minutiae-side layer names unchanged from the first FPP10 runs
    if arm == 'linear_map': h = tf.keras.layers.Dense(M.EMBEDDING_DIM, use_bias=False, name=f'{pre}linear_map')(x)
    else:
        h = tf.keras.layers.Dense(ADAPTER_HIDDEN, activation='gelu', name=f'{pre}mlp_hidden')(x); h = tf.keras.layers.Dropout(ADAPTER_DROPOUT, name=f'{pre}mlp_dropout')(h)
        h = tf.keras.layers.Dense(M.EMBEDDING_DIM, name=f'{pre}mlp_out')(h)
    e = tf.keras.layers.Lambda(lambda z: tf.math.l2_normalize(z, axis=1), name=f'{pre}image_space_embedding' if side == 'minutiae' else 'image_shared_embedding')(h)
    return tf.keras.Model(x, e, name=f'{arm}_adapter' if side == 'minutiae' else f'{arm}_image_adapter')

class CrossModel(tf.keras.Model):
    # f maps the minutiae side (g, two_tower only: the image side) into the shared space; loss = cross-modal BCE (both directions) + LAMBDA_PULL * (1 - cos(f(minutiae_i), g(image_i)))
    def __init__(self, f, g=None, **kw):
        super().__init__(**kw); self.f = f; self.g = g; self.calib = T.CosineCalibration(name='same_finger_probability'); self.bce = tf.keras.losses.BinaryCrossentropy()
        self.t_loss = tf.keras.metrics.Mean(name='loss'); self.t_bce = tf.keras.metrics.Mean(name='bce'); self.t_pull = tf.keras.metrics.Mean(name='pull_loss')
        self.t_acc = tf.keras.metrics.BinaryAccuracy(name='verification_accuracy'); self.t_auc = tf.keras.metrics.AUC(name='verification_auc')
    @property
    def metrics(self): return [self.t_loss, self.t_bce, self.t_pull, self.t_acc, self.t_auc]
    def img(self, I, training): I = tf.cast(I, tf.float32); return I if self.g is None else self.g(I, training=training)
    def call(self, inputs, training=False):
        return self.calib(tf.reduce_sum(self.f(inputs[0], training=training) * self.img(inputs[1], training), axis=1))
    def train_step(self, data):
        (xa, xb, Ia, Ib), y = data; y = tf.cast(tf.reshape(y, [-1]), tf.float32)
        with tf.GradientTape() as tape:
            ma = self.f(xa, training=True); mb = self.f(xb, training=True); Ia = self.img(Ia, True); Ib = self.img(Ib, True)
            pa = self.calib(tf.reduce_sum(ma * Ib, axis=1)); pb = self.calib(tf.reduce_sum(mb * Ia, axis=1)); l_bce = 0.5 * (self.bce(y, pa) + self.bce(y, pb))
            l_pull = 0.5 * (tf.reduce_mean(1.0 - tf.reduce_sum(ma * Ia, axis=1)) + tf.reduce_mean(1.0 - tf.reduce_sum(mb * Ib, axis=1)))
            loss = l_bce + LAMBDA_PULL * l_pull
        tv = self.trainable_variables; self.optimizer.apply_gradients([(g_, v_) for g_, v_ in zip(tape.gradient(loss, tv), tv) if g_ is not None])
        for m_, v_ in ((self.t_loss, loss), (self.t_bce, l_bce), (self.t_pull, l_pull)): m_.update_state(v_)
        p = tf.concat([pa, pb], 0); yy = tf.concat([y, y], 0); self.t_acc.update_state(yy, p); self.t_auc.update_state(yy, p); return {m_.name: m_.result() for m_ in self.metrics}

def build_cross(f, x_example, lr, g=None):
    m = CrossModel(f, g, name='cross_modal'); m((x_example[:1], np.zeros((1, M.EMBEDDING_DIM), np.float32))); m.built = True
    m.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=WEIGHT_DECAY, clipnorm=CLIPNORM)); return m

# ---- closed-form maps (fitted on the TRAINING templates of the repetition: minutiae embedding -> image embedding of the same capture)
def train_index(proto):
    out = []
    for d in MOLF:
        s_ = proto['samples'][d]; out.append(s_[s_.partition == 'train'].sample_index.to_numpy().astype(int) + OFFSET[d])
    return np.concatenate(out)
def fit_ridge(Mt, It, alpha):
    A = Mt.T @ Mt + alpha * np.eye(Mt.shape[1], dtype=np.float64); return np.linalg.solve(A, Mt.T @ It).astype(np.float32)
def fit_procrustes(Mt, It):
    U, _, Vt = np.linalg.svd(Mt.T.astype(np.float64) @ It.astype(np.float64)); return (U @ Vt).astype(np.float32)
def val_eer(Wmap, rep):
    bv = CrossValidation(PROTO[rep][1], OFFSET, I_MOLF[rep], max_pairs=VAL_PAIRS); return bv.score(L.l2n(M0_MOLF[rep][bv.indices] @ Wmap))

# ---- xgboost pair scorer (score, not an embedding)
def pair_features(Mm, Ii): return np.concatenate([Mm * Ii, np.abs(Mm - Ii)], axis=1).astype(np.float32)
def xgb_scores(model, Ea, Eb, pairs, ntree):
    a = np.asarray(pairs['index_a']); b = np.asarray(pairs['index_b']); out = []
    for s_ in range(0, len(a), 50_000):
        out.append(model.predict_proba(pair_features(Ea[a[s_:s_ + 50_000]], Eb[b[s_:s_ + 50_000]]), iteration_range=(0, ntree))[:, 1])
    return np.concatenate(out) if out else np.zeros(0)
print('models ready | linear_map', build_adapter('linear_map').count_params(), '| mlp_adapter', build_adapter('mlp_adapter').count_params(), '| two_tower', 2 * build_adapter('mlp_adapter').count_params(), 'parameters')

## 4. One run = one (arm, repetition)
Every run saves the **cross-modal** scores as its main result (`pair_scores.npz`, `metrics.csv`) and, reported only, `metrics_minutiae_vs_minutiae.csv` (both sides migrated) and `metrics_image_enrol_minutiae_probe.csv` (reverse direction).

In [ ]:
def scores(Ea, Eb, Pp): return {n: (np.asarray(Pp[n]['label'], int), L.pair_scores(Ea[da], Eb[db], Pp[n]).astype(np.float64)) for n, (da, db, _) in P.PAIR_SETS.items()}

def run_arm(arm, rep):
    rd = KDIR / 'runs' / rep / arm
    if L.run_complete(rd): print(arm, rep, 'complete -> skipped (reused)'); return
    global IMG_MOLF
    lrd = P.local_run_dir(KERNEL + ('_smoke' if QUICK_SMOKE_TEST else ''), rep, arm); proto, Pp = PROTO[rep]; seed = proto['seed']; t = L.timer()
    I, extra, emb_m, files_skip, hist = I_EMB[rep], {}, None, ('fit_complete.json', 'best_cross.weights.h5'), None
    tf.keras.backend.clear_session(); gc.collect(); T.set_all_seeds(seed)
    if arm == 'frozen_as_is':
        emb_m = M0_EMB[rep]; extra = {'trained': False, 'training_seconds': 0.0}; LOG.add(step='train_start', arm=arm, repetition=rep, note='no mapping: both models as they are')
    elif arm in ('ridge', 'procrustes'):
        tr = train_index(proto); Mt, It = M0_MOLF[rep][tr], I_MOLF[rep][tr]
        if arm == 'ridge':
            grid = []
            for a_ in RIDGE_ALPHAS:
                W_ = fit_ridge(Mt, It, a_); m_ = val_eer(W_, rep); grid.append({'alpha': a_, 'val_balanced_eer': m_['val_balanced_eer'], 'val_balanced_auc': m_['val_balanced_auc']})
            grid = pd.DataFrame(grid); grid.to_csv(lrd / 'ridge_alpha_selection.csv', index=False); best = grid.sort_values(['val_balanced_eer', 'val_balanced_auc'], ascending=[True, False]).iloc[0]
            Wmap = fit_ridge(Mt, It, float(best.alpha)); extra = {'trained': True, 'closed_form': True, 'ridge_alpha': float(best.alpha)}
            print('ridge alpha selection (validation):', grid.round(4).to_dict('records'))
        else:
            Wmap = fit_procrustes(Mt, It); extra = {'trained': True, 'closed_form': True, 'orthogonality_error': float(np.abs(Wmap.T @ Wmap - np.eye(len(Wmap))).max())}
        extra.update(training_seconds=t(), training_templates=int(len(tr))); np.save(lrd / 'map_matrix.npy', Wmap)
        emb_m = {d: L.l2n(M0_EMB[rep][d] @ Wmap) for d in L.DATASETS}
        LOG.add(step='train_start', arm=arm, repetition=rep, **{k: v for k, v in extra.items() if k != 'trained'})
    elif arm == 'xgboost':
        import xgboost as xgb
        ssf = float(proto['manifest']['same_subject_impostor_fraction'])
        smp = P.MixedPairs(P.sources(Pp, P.TRAIN_SOURCES, OFFSET), PAIR_BATCH, True, seed, XGB_TRAIN_PAIRS, ssf); ia_, ib_, y_ = [], [], []
        for b_ in range(len(smp)):
            a1, b1, yy = smp.batch(b_); ia_.append(np.asarray(a1)); ib_.append(np.asarray(b1)); y_.append(np.asarray(yy))
        ia_, ib_, y_ = np.concatenate(ia_), np.concatenate(ib_), np.concatenate(y_)
        Xtr = np.concatenate([pair_features(M0_MOLF[rep][ia_], I_MOLF[rep][ib_]), pair_features(M0_MOLF[rep][ib_], I_MOLF[rep][ia_])]); ytr = np.concatenate([y_, y_])
        model = xgb.XGBClassifier(**XGB, random_state=int(seed) % (2 ** 31)); model.fit(Xtr, ytr, verbose=False)
        bvx = P.BalancedValidation(Pp, OFFSET, max_pairs=VAL_PAIRS); grid = []
        for nt in [n for n in XGB_TREE_GRID if n <= XGB['n_estimators']]:
            ys, ss = [], []
            for n_, (a1, b1, yy) in bvx.sets.items():
                s_ = model.predict_proba(pair_features(M0_MOLF[rep][a1], I_MOLF[rep][b1]), iteration_range=(0, nt))[:, 1]; ys.append(yy); ss.append(s_)
            e_, au_, _ = P.balanced_eer(ys, ss); grid.append({'trees': nt, 'val_balanced_eer': e_, 'val_balanced_auc': au_})
        grid = pd.DataFrame(grid); grid.to_csv(lrd / 'xgboost_tree_selection.csv', index=False); NT = int(grid.sort_values(['val_balanced_eer', 'val_balanced_auc'], ascending=[True, False]).iloc[0].trees)
        model.save_model(str(lrd / 'xgboost_pair_scorer.json'))
        extra = {'trained': True, 'xgboost_trees': NT, 'training_pairs': int(len(ytr)), 'training_seconds': t(), 'score': 'probability, not an embedding'}
        LOG.add(step='train_start', arm=arm, repetition=rep, xgboost_trees=NT, training_pairs=int(len(ytr)))
        S = {n: (np.asarray(Pp[n]['label'], int), xgb_scores(model, M0_EMB[rep][da], I_EMB[rep][db], Pp[n], NT).astype(np.float64)) for n, (da, db, _) in P.PAIR_SETS.items()}
    else:          # trained mappings: linear_map, mlp_adapter, two_tower
        f = build_adapter('mlp_adapter' if arm == 'two_tower' else arm); g = build_adapter('mlp_adapter', side='image') if arm == 'two_tower' else None
        X_molf = M0_MOLF[rep]; IMG_MOLF = X_molf
        model = build_cross(f, X_molf, LR_BY_ARM[arm], g); params = T.count_params(f); ssf = float(proto['manifest']['same_subject_impostor_fraction'])
        seq = CrossPairSeq(P.MixedPairs(P.sources(Pp, P.TRAIN_SOURCES, OFFSET), PAIR_BATCH, True, seed, TRAIN_PAIRS_PER_EPOCH, ssf), seed, 'adapter', X_molf, I_MOLF[rep])
        bv = CrossValidation(Pp, OFFSET, I_MOLF[rep], max_pairs=VAL_PAIRS, g=g)
        LOG.add(step='train_start', arm=arm, repetition=rep, learning_rate=LR_BY_ARM[arm], train_composition=json.dumps(seq.s.composition), validation_pairs=json.dumps(bv.composition), **params)
        ckpt = lrd / 'best_cross.weights.h5'; hist_csv = lrd / 'training_history.csv'; fit_marker = lrd / 'fit_complete.json'
        if ckpt.is_file() and fit_marker.is_file(): train_s = json.loads(fit_marker.read_text())['training_seconds']
        else:
            model.fit(seq, epochs=EPOCHS_BY_ARM[arm], callbacks=[EpochClock(), BalancedEERCallback(f, bv, ckpt, hist_csv)], verbose=2); train_s = t(); fit_marker.write_text(json.dumps({'training_seconds': train_s}))
        model.load_weights(ckpt); hist = pd.read_csv(hist_csv)
        b_i, _ = P.BalancedEERStopping.replay(hist, EPOCHS_BY_ARM[arm], patience=PATIENCE, lr_patience=LR_PATIENCE, lr_factor=LR_FACTOR); best = hist.iloc[b_i]
        emb_m = {d: L.l2n(T.embed(f, M0_EMB[rep][d], EMBED_BATCH)) for d in L.DATASETS}; f.save_weights(lrd / 'adapter.weights.h5')
        if g is not None:
            I = {d: L.l2n(T.embed(g, I_EMB[rep][d], EMBED_BATCH)) for d in L.DATASETS}; g.save_weights(lrd / 'image_adapter.weights.h5')
            np.savez_compressed(lrd / 'image_side_embeddings_float32.npz', **{d: e.astype(np.float32) for d, e in I.items()})
        extra = {'trained': True, 'learning_rate': LR_BY_ARM[arm], 'best_epoch': int(best.epoch), 'epochs_run': int(len(hist)), 'val_balanced_eer_at_best': float(best.val_balanced_eer),
                 'training_seconds': train_s, 'seconds_per_epoch': float(hist['epoch_seconds'].mean()) if 'epoch_seconds' in hist else float('nan'), **params}
    if arm != 'xgboost':
        same = float(np.mean(np.sum(np.concatenate([emb_m[d] for d in MOLF]) * np.concatenate([I[d] for d in MOLF]), 1)))
        np.savez_compressed(lrd / 'embeddings_float32.npz', **{d: e.astype(np.float32) for d, e in emb_m.items()})          # migrated templates
        S = scores(emb_m, I, Pp)
        others = [('metrics_minutiae_vs_minutiae.csv', scores(emb_m, emb_m, Pp), 'minutiae enrolled vs minutiae probe (both mapped)'),
                  ('metrics_image_enrol_minutiae_probe.csv', scores(I, emb_m, Pp), 'image enrolled vs minutiae probe')]
        if arm == 'two_tower': others.append(('metrics_image_vs_image_after.csv', scores(I, I, Pp), 'image vs image after the image adapter'))
        for fname, S2, what in others: pd.DataFrame(P.evaluate(S2, KERNEL, arm, rep, {'comparison': what})[0]).to_csv(lrd / fname, index=False)
    else: same = float('nan')
    (lrd / 'model_config.json').write_text(json.dumps({'arm': arm, 'kernel': KERNEL, 'input': 'frozen FPP09 minutiae embeddings + frozen FPP04L image embeddings', **{k: v for k, v in extra.items() if not isinstance(v, float) or k in ('ridge_alpha',)}}, indent=2, default=str))
    rows, thr = P.evaluate(S, KERNEL, arm, rep, {'comparison': 'minutiae enrolled vs image probe', 'cos_same_capture': same, **extra})
    P.save_run(lrd, S, rows, thr, {'kernel': KERNEL, 'arm': arm, 'repetition': rep, 'seed': seed, 'cos_same_capture': same, **extra})
    n_files = P.publish_run(lrd, rd, skip=files_skip)
    if hist is not None: P.fig_training_balanced(hist, f'{arm} {rep[:6]} — stopping on cross-modal source-balanced validation EER', FIG / 'runs' / rep / f'{arm}_training.png')
    v = [r_ for r_ in rows if r_['set'] == P.SELECTION_SET][0]
    LOG.add(step='train_done', arm=arm, repetition=rep, status='ok', minutes=round(t() / 60, 1), files_to_drive=n_files, cos_same_capture=round(same, 4),
            val_balanced_eer=round(100 * v['eer'], 2), val_balanced_auc=round(100 * v['roc_auc'], 2), val_balanced_f1=round(100 * v['f1_macro'], 2))
    print(f"{arm} {rep[:6]}: cross-modal balanced validation EER {100 * v['eer']:.2f} % | cos(same capture) {same:.3f}")

## 5. Run every arm for every repetition (finished runs are reused; every arm takes seconds to minutes)

In [ ]:
for arm in ARMS_TO_RUN:
    for rep in REPS_TO_RUN:
        try: run_arm(arm, rep)
        except (tf.errors.ResourceExhaustedError, tf.errors.InternalError) as exc:
            LOG.add(step='train_failed', arm=arm, repetition=rep, status='gpu_error', error=repr(exc)[:400]); print('GPU error', arm, rep, repr(exc)[:200]); tf.keras.backend.clear_session(); gc.collect()
        except Exception as exc:
            LOG.add(step='train_failed', arm=arm, repetition=rep, status='error', error=repr(exc)[:400]); print('FAILED', arm, rep, exc)
print(LOG.frame().tail(15).to_string())

## 6. Decision on VALIDATION only (cross-modal); references; test and FingerDOS reported, not used; migration matrix

In [ ]:
complete = [a for a in ARMS if all(L.run_complete(KDIR / 'runs' / r / a) for r in REPS_TO_RUN)]
if not complete: raise RuntimeError('no finished arm: run section 5 first')
results = pd.concat([pd.read_csv(KDIR / 'runs' / r / a / 'metrics.csv').assign(kernel=a) for a in complete for r in REPS_TO_RUN], ignore_index=True)
HIST = [pd.read_csv(KDIR / 'runs' / r / a / 'training_history.csv').assign(arm=a, repetition=r) for a in complete for r in REPS_TO_RUN if (KDIR / 'runs' / r / a / 'training_history.csv').is_file()]
histories = pd.concat(HIST, ignore_index=True) if HIST else pd.DataFrame()
def ref(label, run_dirs):
    rd0 = next(iter(run_dirs.values())); rows = pd.DataFrame(P.reference_rows(label, rd0.parents[2], arm=rd0.name, reps=REPS_TO_RUN)); rows['kernel'] = label; return rows
ref_img = ref(IMG_LABEL, IMG_RUN); ref_min = ref(MIN_LABEL, MIN_RUN)
FPP03_DIR = P.kernel_dir(ws, 'FPP03'); ref_b = pd.DataFrame(P.reference_rows(BOZ, FPP03_DIR, arm='BOZORTH3', reps=REPS_TO_RUN)); ref_b['kernel'] = BOZ
ALLR = pd.concat([results, ref_img, ref_min, ref_b], ignore_index=True); SHOW = complete + [IMG_LABEL, MIN_LABEL, BOZ]

# ---- 6.1 decision: cross-modal validation only
WINNER, decision = P.decide(ALLR, complete)
val_table = P.validation_table(ALLR, SHOW)
print('DECISION RULE (cross-modal):', P.DECISION_RULE); display(decision.round(3))
print('Validation, mean ± std (%) — arms: minutiae enrolled vs image probe; reference rows: image vs image (ceiling), minutiae vs minutiae and BOZORTH3 (old system):'); display(val_table)
print('WINNER (validation only):', WINNER)
cos_same = results[results.set == P.SELECTION_SET].groupby('kernel').cos_same_capture.agg(['mean', 'std']).reindex(complete); print('cos(minutiae embedding, image embedding of the same capture):'); display(cos_same.round(4))

# ---- 6.2 test and FingerDOS — reported only
paper_all, snum_all = P.paper_table(ALLR, SHOW, sets=['MOLF validation (balanced)', 'MOLF test (balanced)', 'DB1_test', 'DB2_test', 'DB1xDB2_test', 'FingerDOS_test'])
test_table = P.wide_table(snum_all, SHOW, sets=['MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'], metrics_=('roc_auc', 'eer', 'fmr', 'fnmr', 'f1_macro'))
print('Test and FingerDOS incl. F1 (frozen validation-EER threshold; reported, not used for the decision):'); display(test_table)
paired = pd.concat([P.paired_differences(ALLR, a, [x for x in complete if x != a] + [IMG_LABEL, MIN_LABEL]) for a in complete if a != 'frozen_as_is'], ignore_index=True) if len(complete) > 1 else pd.DataFrame()

# ---- 6.3 reported only: minutiae vs minutiae after migration, reverse direction, and the migration matrix (MOLF test balanced + FingerDOS)
def extra_rows(fname, label_fmt):
    out = [pd.read_csv(KDIR / 'runs' / r / a / fname).assign(kernel=label_fmt.format(a)) for a in complete for r in REPS_TO_RUN if (KDIR / 'runs' / r / a / fname).is_file()]
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()
MM = extra_rows('metrics_minutiae_vs_minutiae.csv', '{} — minutiae vs minutiae'); RV = extra_rows('metrics_image_enrol_minutiae_probe.csv', '{} — image enrolled vs minutiae probe')
II = pd.DataFrame()          # two_tower's image adapter is used only for the first-login match: image vs image always uses the unchanged image model
extra_all, extra_num = P.paper_table(pd.concat([MM, RV, II, ref_min, ref_img], ignore_index=True), list(MM.kernel.unique()) + list(RV.kernel.unique()) + (list(II.kernel.unique()) if len(II) else []) + [MIN_LABEL, IMG_LABEL],
                                      sets=['MOLF validation (balanced)', 'MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'])
print('Reported only — matching after migration in the other combinations:'); display(extra_all)
def cell(frame, kernel, set_):
    if not len(frame) or 'set' not in frame.columns: return '—'
    v = frame[(frame.kernel == kernel) & (frame.set == set_)].eer * 100; return f'{v.mean():.2f} ± {v.std():.2f}' if len(v) else '—'
mat = []
for set_ in ('MOLF test (balanced)', 'FingerDOS_test'):
    mat += [{'set': set_, 'enrolled \\ probe': 'minutiae (migrated)', 'minutiae probe': cell(MM, f'{WINNER} — minutiae vs minutiae', set_), 'image probe': cell(results, WINNER, set_)},
            {'set': set_, 'enrolled \\ probe': 'image', 'minutiae probe': cell(RV, f'{WINNER} — image enrolled vs minutiae probe', set_),
             'image probe': cell(ref_img, IMG_LABEL, set_)},
            {'set': set_, 'enrolled \\ probe': 'old system', 'minutiae probe': f"FPP09 {cell(ref_min, MIN_LABEL, set_)} | BOZORTH3 {cell(ref_b, BOZ, set_)}", 'image probe': '—'}]
migration_matrix = pd.DataFrame(mat); print(f'MIGRATION MATRIX — EER (%), winner {WINNER} (reported, not used for the decision):'); display(migration_matrix)
tcols = ['training_seconds', 'best_epoch', 'epochs_run', 'seconds_per_epoch', 'parameters_total', 'ridge_alpha', 'xgboost_trees']
tsrc = results[results.set == P.SELECTION_SET]; timing = tsrc.groupby('kernel')[[c_ for c_ in tcols if c_ in tsrc.columns]].agg(['mean', 'std']).reindex(complete)
timing.columns = [f'{a}_{b}' for a, b in timing.columns]; print('Training time and selected settings (reported only):'); display(timing.round(4))

## 6b. Bias check of the threshold (cross-modal): 5 calibration hold-out subjects vs validation (MOLF test, balanced)

In [ ]:
CB = []
for a in [x for x in complete if x != 'xgboost']:          # xgboost gives a pair score, not embeddings: its threshold check is the validation rule only
    for r in REPS_TO_RUN:
        rd = KDIR / 'runs' / r / a; proto, Pp = PROTO[r]; Em = L.load_npz(rd / 'embeddings_float32.npz')
        Ii = L.load_npz(rd / 'image_side_embeddings_float32.npz') if (rd / 'image_side_embeddings_float32.npz').is_file() else I_EMB[r]
        E = {d: (Em[d], Ii[d]) for d in L.DATASETS}               # (enrolled minutiae, probe image) per dataset
        c_ = P.calibration_check(P.load_scores(rd), E, P.calibration_pairs(proto), pair_score=lambda Ea, Eb, p: L.pair_scores(Ea[0], Eb[1], p)); CB.append({'arm': a, 'repetition': r, **c_})
calib = pd.DataFrame(CB); cs = calib.drop(columns='repetition').groupby('arm').agg(['mean', 'std'])
calib_table = pd.DataFrame({'arm': cs.index,
    'threshold validation': [f"{a_:.3f} ± {b_:.3f}" for a_, b_ in zip(cs[('threshold_validation', 'mean')], cs[('threshold_validation', 'std')])],
    'threshold calibration (5 subj.)': [f"{a_:.3f} ± {b_:.3f}" for a_, b_ in zip(cs[('threshold_calibration', 'mean')], cs[('threshold_calibration', 'std')])],
    **{f'test {m.upper()} {tag} (%)': (100 * cs[(f'test_{m}_{tag}', 'mean')]).round(2).to_numpy() for m in ('fmr', 'fnmr', 'f1') for tag in ('validation', 'calibration')}}).reset_index(drop=True)
display(calib_table)

## 7. Figures

In [ ]:
MR = {a: [KDIR / 'runs' / r / a for r in REPS_TO_RUN] for a in complete}
MR[IMG_LABEL] = [IMG_RUN[r] for r in REPS_TO_RUN]; MR[MIN_LABEL] = [MIN_RUN[r] for r in REPS_TO_RUN]
P.standard_figures(FIG, MR, sets=P.PAPER_SETS)
P.fig_metric_bars(snum_all, SHOW, FIG / 'metric_bars_validation.png', sets=['MOLF validation (balanced)'], title='Cross-modal validation (decision set) with references')
P.fig_metric_bars(snum_all, SHOW, FIG / 'metric_bars_test_reported.png', sets=['MOLF test (balanced)', 'DB1xDB2_test', 'FingerDOS_test'], title='Test and FingerDOS (reported, not used for the decision)')
v = ALLR[ALLR.set == P.SELECTION_SET].groupby('kernel').eer.agg(['mean', 'std']) * 100
fig, ax = plt.subplots(figsize=(9, 4)); xs = complete
ax.bar(range(len(xs)), v.loc[xs, 'mean'], yerr=v.loc[xs, 'std'], capsize=4, color=[P.COLORS[0] if a == WINNER else '0.7' for a in xs])
for lab, col, ls in ((IMG_LABEL, 'C2', '--'), (MIN_LABEL, 'C1', ':'), (BOZ, '0.4', '-.')):
    if lab in v.index: ax.axhline(v.loc[lab, 'mean'], color=col, ls=ls, lw=1.2, label=f"{lab}: {v.loc[lab, 'mean']:.2f} %")
ax.set_xticks(range(len(xs))); ax.set_xticklabels(xs); ax.set(ylabel='validation EER, source-balanced (%)', title='Minutiae enrolled vs image probe, by arm (mean ± std)'); ax.legend(fontsize=7); ax.grid(alpha=.3, axis='y')
P._save(fig, FIG / 'migration_arms.png')
r0 = REPS_TO_RUN[0]; fig, axs = plt.subplots(1, len(complete), figsize=(4 * len(complete), 3.5), squeeze=False)
for ax, a in zip(axs[0], complete):
    Sx = P.load_scores(KDIR / 'runs' / r0 / a); y, s = P.group_scores(Sx, 'MOLF test (pooled)')
    ax.hist(s[y == 1], bins=50, alpha=.6, density=True, label='same finger'); ax.hist(s[y == 0], bins=50, alpha=.6, density=True, label='different finger'); ax.set(title=a, xlabel='cosine'); ax.legend(fontsize=7)
fig.suptitle(f'Cross-modal scores, MOLF test ({r0[:6]})', fontsize=10); P._save(fig, FIG / 'cross_modal_score_distributions.png'); print('figures in', FIG)

## 8. Manifest and workbook

In [ ]:
if not QUICK_SMOKE_TEST and set(ARMS) <= set(complete):
    wfile = {'linear_map': ('adapter.weights.h5',), 'mlp_adapter': ('adapter.weights.h5',), 'two_tower': ('adapter.weights.h5', 'image_adapter.weights.h5', 'image_side_embeddings_float32.npz'),
             'ridge': ('map_matrix.npy',), 'procrustes': ('map_matrix.npy',), 'xgboost': ('xgboost_pair_scorer.json',), 'frozen_as_is': ()}[WINNER]
    files = tuple(wfile) + ('model_config.json',) * (WINNER not in ('frozen_as_is',) and (KDIR / 'runs' / REPS_TO_RUN[0] / WINNER / 'model_config.json').is_file()) + (('embeddings_float32.npz',) if WINNER != 'xgboost' else ()) + ('thresholds.json', 'pair_scores.npz')
    manifest = {'kernel': KERNEL, 'kernel_name': P.FPP_KERNELS[KERNEL], 'winning_arm': WINNER, 'cross_modal': CONFIG['cross_modal'], 'image_model': {r: L.rel(ws, d) for r, d in IMG_RUN.items()},
                'minutiae_start': {r: L.rel(ws, d) for r, d in MIN_RUN.items()}, 'minutiae_config': M_CFG, 'loss': CONFIG['loss'], 'decision_rule': P.DECISION_RULE,
                'decision': decision.to_dict('records'), 'migration_matrix': migration_matrix.to_dict('records'), 'threshold_rule': P.THRESHOLD_RULE,
                'files': {r: {k: L.rel(ws, KDIR / 'runs' / r / WINNER / k) for k in files} for r in REPS_TO_RUN}, 'module': 'fpp10_minutiae.py', 'fpp_lib_version': P.VERSION}
    (KDIR / 'winner_manifest.json').write_text(json.dumps(manifest, indent=2, default=str)); (KDIR / 'fpp10_minutiae.py').write_text(MOD_PATH.read_text())
    print('winner_manifest.json written | winner:', WINNER)
SHEETS = {'Decision (validation)': decision, 'Validation table': val_table, 'Migration matrix': migration_matrix, 'Test+FingerDOS (reported)': test_table,
          'Other combinations (reported)': extra_all, 'cos same capture': cos_same.reset_index(), 'Time (reported)': timing.reset_index(), 'All sets (mean ± std)': paper_all,
          'Paired differences': paired, 'Calibration Bias Check': calib_table, 'Calibration (per rep)': calib, 'Results (per rep)': P.to_percent(ALLR),
          'Training history': histories, 'Config': pd.DataFrame([{k: json.dumps(v_, default=str) for k, v_ in CONFIG.items()}]).T.reset_index(), 'Run log': LOG.frame()}
L.write_workbook(KDIR / 'FPP10_CrossModal_Frozen_Encoders_Results.xlsx', SHEETS)
for name, df in (('decision', decision), ('validation_table', val_table), ('migration_matrix', migration_matrix), ('test_fingerdos_reported', test_table),
                 ('other_combinations_reported', extra_all), ('paired_differences', paired), ('calibration_bias_check', calib_table), ('results_per_repetition', P.to_percent(ALLR))):
    df.to_csv(KDIR / f'FPP10_{name}.csv', index=False)
print('FPP10 complete | winner on validation:', WINNER)